# WaterOpt


In [ ]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")

# USGS NWIS API endpoint
USGS_API_URL = "https://waterdata.usgs.gov/nwis/qw"

# Walnut Creek near Madrid, Iowa (USGS Station ID: 05481000)
# If this doesn't work, we can use Raccoon River station instead
STATION_ID = "05481000"

print(f"USGS Station: {STATION_ID}")
print("Fetching streamflow data...")

In [ ]:
def fetch_usgs_daily_streamflow(site_id, start_date, end_date):
    params = {
        "format": "json",
        "sites": site_id,
        "startDT": start_date,
        "endDT": end_date,
        "parameterCd": "00060",  # discharge
        "siteStatus": "all",
    }
    resp = requests.get("https://waterservices.usgs.gov/nwis/dv/", params=params)
    resp.raise_for_status()
    payload = resp.json()

    series = payload.get("value", {}).get("timeSeries", [])
    if not series:
        return pd.DataFrame(columns=["flow"], index=pd.to_datetime([]))

    values = series[0].get("values", [])[0].get("value", [])
    df = pd.DataFrame(values)
    if df.empty:
        return pd.DataFrame(columns=["flow"], index=pd.to_datetime([]))

    df = df.rename(columns={"dateTime": "date", "value": "flow"})
    df["date"] = pd.to_datetime(df["date"])
    df["flow"] = pd.to_numeric(df["flow"], errors="coerce")
    return df.set_index("date")[["flow"]]


# Fetch daily streamflow for Walnut Creek station
baseline_daily = fetch_usgs_daily_streamflow(STATION_ID, "1991-01-01", "2020-12-31")
recent_daily = fetch_usgs_daily_streamflow(STATION_ID, "2022-01-01", "2025-12-31")

# Convert daily values to monthly average streamflow
baseline_monthly = baseline_daily.resample("ME").mean()
recent_monthly = recent_daily.resample("ME").mean()

baseline_monthly["month"] = baseline_monthly.index.month
monthly_thresholds = (
    baseline_monthly.groupby("month")["flow"]
    .agg(
        baseline_median="median",
        threshold_70=lambda x: x.quantile(0.70),
        threshold_75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)
monthly_thresholds["month_name"] = pd.to_datetime(
    monthly_thresholds["month"], format="%m"
).dt.strftime("%b")

recent_monthly["month"] = recent_monthly.index.month
recent_with_thresholds = recent_monthly.join(
    monthly_thresholds.set_index("month"),
    on="month",
)

print("Baseline monthly thresholds (1991-2020):")
print(monthly_thresholds)

print("\nRecent monthly streamflow with matched baseline thresholds (2022-2025):")
print(recent_with_thresholds.head(24))

In [ ]:
import matplotlib.pyplot as plt

threshold_70_series = baseline_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_70"]
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    baseline_monthly.index,
    baseline_monthly["flow"],
    label="Baseline monthly flow (1991-2020)",
)
ax.plot(
    baseline_monthly.index,
    threshold_70_series,
    color="red",
    lw=2,
    label="Monthly 70% threshold",
)
ax.set_ylabel("Flow (cfs)")
ax.set_xlabel("Date")
ax.set_ylabel("Flow")
ax.set_title("Baseline Monthly Flow (1991-2020) and Monthly 70% Thresholds")
ax.legend()
ax.grid(True)
plt.tight_layout()
import matplotlib.pyplot as plt

threshold_70_series = baseline_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_70"]
)

threshold_75_series = baseline_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_75"]
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    baseline_monthly.index,
    baseline_monthly["flow"],
    label="Baseline monthly flow (1991-2020)",
)
ax.plot(
    baseline_monthly.index,
    threshold_70_series,
    color="red",
    lw=2,
    label="Monthly 70% threshold",
)
ax.plot(
    baseline_monthly.index,
    threshold_75_series,
    color="orange",
    lw=2,
    label="Monthly 75% threshold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Flow")
ax.set_title(
    "Baseline Monthly Flow (1991-2020) and Monthly Thresholds for Walnut Creek"
)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Map thresholds to recent monthly data
threshold_70_series_recent = recent_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_70"]
)

threshold_75_series_recent = recent_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_75"]
)

import matplotlib.pyplot as plt

# Map thresholds to recent monthly data
threshold_70_series_recent = recent_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_70"]
)

threshold_75_series_recent = recent_monthly["month"].map(
    monthly_thresholds.set_index("month")["threshold_75"]
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    recent_monthly.index,
    recent_monthly["flow"],
    label="Recent monthly flow (2022-2025)",
)
ax.plot(
    recent_monthly.index,
    threshold_70_series_recent,
    color="red",
    lw=2,
    label="Monthly 70% threshold",
)
ax.plot(
    recent_monthly.index,
    threshold_75_series_recent,
    color="orange",
    lw=2,
    label="Monthly 75% threshold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Flow (cfs)")
ax.set_title("Recent Monthly Flow (2022-2025) and Monthly Thresholds for Walnut Creek")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()
ax.plot(
    recent_monthly.index,
    recent_monthly["flow"],
    label="Recent monthly flow (2022-2025)",
)
ax.plot(
    recent_monthly.index,
    threshold_70_series_recent,
    color="red",
    lw=2,
    label="Monthly 70% threshold",
)
ax.plot(
    recent_monthly.index,
    threshold_75_series_recent,
    color="orange",
    lw=2,
    label="Monthly 75% threshold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Flow")
ax.set_title("Recent Monthly Flow (2022-2025) and Monthly Thresholds for Walnut Creek")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Define seasons based on months


def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"


# Add season column to baseline_daily
baseline_daily["season"] = baseline_daily.index.month.map(get_season)

# Define colors for seasons
season_colors = {"Winter": "blue", "Spring": "green", "Summer": "red", "Fall": "orange"}

# Plot daily flow color-coded by season
fig, ax = plt.subplots(figsize=(14, 7))
for season, color in season_colors.items():
    subset = baseline_daily[baseline_daily["season"] == season]
    ax.scatter(subset.index, subset["flow"], color=color, label=season, s=1, alpha=0.6)

# Plot monthly average on top
ax.plot(
    baseline_monthly.index,
    baseline_monthly["flow"],
    color="black",
    linewidth=2,
    label="Monthly Average",
)

ax.set_xlabel("Date")
ax.set_ylabel("Flow (cfs)")
ax.set_title(
    "Baseline Daily Flow (1991-2020) Color-Coded by Season with Monthly Average Overlay"
)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## USGS Hydrologic Drought Definition

USGS defines **hydrologic drought** using **streamflow percentiles** relative to the historical record for the same day of year (via the [WaterWatch](https://waterwatch.usgs.gov) program):

| Category | Streamflow Percentile |
|---|---|
| Normal | 25th – 75th |
| Below normal | 10th – 24th |
| **Much below normal (drought)** | **< 10th percentile** |
| **Severe drought** | **< 5th percentile** |

A **drought period** is declared when daily streamflow falls **below the 10th percentile** of the historical flow for that calendar day, persisting for **at least 7 consecutive days**.

Thresholds are computed from the 1991–2020 baseline (the current USGS 30-year normal period).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- Step 1: Compute day-of-year percentile thresholds from the 1991-2020 baseline ---
# Use a 15-day rolling window around each day of year to smooth percentile estimates

doy_thresholds = {}
baseline_daily_clean = baseline_daily.dropna(subset=["flow"])
baseline_daily_clean["doy"] = baseline_daily_clean.index.day_of_year

for doy in range(1, 367):
    # 15-day window (±7 days) to get enough data per DOY
    window_doys = [(doy - 7 + d) % 365 + 1 for d in range(15)]
    subset = baseline_daily_clean[baseline_daily_clean["doy"].isin(window_doys)]["flow"]
    doy_thresholds[doy] = {
        "p10": subset.quantile(0.10),
        "p25": subset.quantile(0.25),
    }

threshold_df = pd.DataFrame(doy_thresholds).T
threshold_df.index.name = "doy"

# --- Step 2: Classify each day in baseline as drought / below-normal / normal ---
df = baseline_daily_clean.copy()
df["doy"] = df.index.day_of_year
df["p10"] = df["doy"].map(threshold_df["p10"])
df["p25"] = df["doy"].map(threshold_df["p25"])

df["drought"] = df["flow"] < df["p10"]  # much below normal
df["below_normal"] = (df["flow"] >= df["p10"]) & (df["flow"] < df["p25"])

# --- Step 3: Identify drought PERIODS (≥7 consecutive drought days) ---


def find_drought_periods(series, min_days=7):
    """Return list of (start, end) date pairs for runs of True lasting >= min_days."""
    periods = []
    in_drought = False
    start = None
    for date, val in series.items():
        if val and not in_drought:
            in_drought = True
            start = date
        elif not val and in_drought:
            in_drought = False
            if (date - start).days >= min_days:
                periods.append((start, date))
    if in_drought and (series.index[-1] - start).days >= min_days:
        periods.append((start, series.index[-1]))
    return periods


drought_periods = find_drought_periods(df["drought"], min_days=7)

print(
    f"Found {len(drought_periods)} drought periods (flow < 10th percentile for ≥7 days)\n"
)
print(f"{'Start':12s}  {'End':12s}  {'Duration (days)':>15s}")
print("-" * 45)
for start, end in drought_periods:
    print(f"{start.date()}  {end.date()}  {(end - start).days:>15d}")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Shade drought periods
for start, end in drought_periods:
    ax.axvspan(start, end, color="red", alpha=0.25, linewidth=0)

# Plot daily flow
ax.plot(
    df.index, df["flow"], color="steelblue", lw=0.8, label="Daily flow (cfs)", zorder=3
)

# Plot smoothed p10 threshold (7-day rolling)
p10_series = df["p10"]
ax.plot(
    df.index,
    p10_series,
    color="darkred",
    lw=1.2,
    linestyle="--",
    label="10th pct. threshold (drought)",
    zorder=4,
)

ax.set_yscale("log")
ax.set_xlabel("Date")
ax.set_ylabel("Flow (cfs, log scale)")
ax.set_title(
    "Walnut Creek Daily Streamflow (1991–2020) with USGS Drought Periods Highlighted\n(red shading = flow < 10th percentile for ≥7 consecutive days)"
)

drought_patch = mpatches.Patch(
    color="red", alpha=0.4, label=f"Drought period ({len(drought_periods)} events)"
)
ax.legend(handles=[ax.lines[0], ax.lines[1], drought_patch])
ax.grid(True, which="both", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()